In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Load the datasets
customers_df = pd.read_csv(r'C:\Users\WIN-11\Desktop\Zeotap Internship\data\Customers.csv')
products_df = pd.read_csv(r'C:\Users\WIN-11\Desktop\Zeotap Internship\data\Products.csv')
transactions_df = pd.read_csv(r'C:\Users\WIN-11\Desktop\Zeotap Internship\data\Transactions.csv')

# Merge transactions with customer and product data
transactions_df = transactions_df.merge(customers_df[['CustomerID', 'Region']], on='CustomerID', how='left')
transactions_df = transactions_df.merge(products_df[['ProductID', 'Category', 'Price']], on='ProductID', how='left')

# Feature engineering: Aggregate transaction data by customer
customer_features = transactions_df.groupby('CustomerID').agg(
    total_spent=('TotalValue', 'sum'),
    avg_spent=('TotalValue', 'mean'),
    total_items=('Quantity', 'sum'),
    unique_products=('ProductID', 'nunique'),
    region=('Region', 'first')  # Assuming a customer belongs to one region
).reset_index()

# Normalize customer features
scaler = StandardScaler()
customer_features[['total_spent', 'avg_spent', 'total_items', 'unique_products']] = scaler.fit_transform(
    customer_features[['total_spent', 'avg_spent', 'total_items', 'unique_products']]
)

# Display the customer features for inspection
customer_features.head()


,CustomerID,total_spent,avg_spent,total_items,unique_products,region
0,C0001,-0.061701,-0.070263,-0.122033,0.050047,South America
1,C0002,-0.877744,-0.934933,-0.448000,-0.424204,Asia
2,C0003,-0.405857,-0.026271,0.203934,-0.424204,South America
3,C0004,1.032547,-0.076769,1.670787,1.472798,South America
4,C0005,-0.783929,-0.040028,-0.936951,-0.898455,Asia


In [4]:
# Generate feature vectors for customers (excluding 'CustomerID' and 'Region' columns)
customer_feature_matrix = customer_features.drop(columns=['CustomerID', 'region'])

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(customer_feature_matrix)

# Convert similarity matrix into a DataFrame for easy manipulation
cosine_sim_df = pd.DataFrame(cosine_sim, index=customer_features['CustomerID'], columns=customer_features['CustomerID'])

# Display similarity matrix (Optional)
print(cosine_sim_df.head())


CustomerID     C0001     C0002     C0003     C0004     C0005     C0006  \
CustomerID                                                               
C0001       1.000000  0.666229 -0.191190 -0.474670  0.491530 -0.523419   
C0002       0.666229  1.000000  0.530239 -0.631647  0.707054 -0.666999   
C0003      -0.191190  0.530239  1.000000 -0.458869  0.539681 -0.052562   
C0004      -0.474670 -0.631647 -0.458869  1.000000 -0.991708 -0.126452   
C0005       0.491530  0.707054  0.539681 -0.991708  1.000000  0.042298   

CustomerID     C0007     C0008     C0009     C0010  ...     C0191     C0192  \
CustomerID                                          ...                       
C0001       0.119316  0.056557  0.767454  0.527378  ...  0.913952  0.770811   
C0002       0.226633 -0.218985  0.959115  0.970414  ...  0.889716  0.957535   
C0003       0.437948 -0.511323  0.381493  0.607178  ...  0.103768  0.411976   
C0004      -0.877216  0.841217 -0.763892 -0.465926  ... -0.485653 -0.789144   
C0005  

In [5]:
# Prepare an empty dictionary to store recommendations
lookalike_dict = {}

# Loop through each of the first 20 customers (C0001 to C0020)
for cust_id in customer_features['CustomerID'][:20]:
    # Get similarity scores for the current customer with all others
    similarity_scores = cosine_sim_df[cust_id]
    
    # Sort similarity scores in descending order (ignoring the self-similarity)
    similar_customers = similarity_scores.drop(cust_id).sort_values(ascending=False).head(3)
    
    # Store the top 3 similar customers and their similarity scores
    lookalike_dict[cust_id] = list(zip(similar_customers.index, similar_customers.values))

# Display the lookalike dictionary for inspection
lookalike_dict

{'C0001': [('C0164', 0.9717747353031432),
  ('C0137', 0.9543958286644648),
  ('C0103', 0.9461004687085581)],
 'C0002': [('C0029', 0.9995432945857693),
  ('C0031', 0.9978269016160849),
  ('C0077', 0.9958019717753117)],
 'C0003': [('C0027', 0.8389931643038906),
  ('C0176', 0.8279582423614589),
  ('C0073', 0.7652715881378696)],
 'C0004': [('C0075', 0.9975621414877183),
  ('C0175', 0.9930015793963254),
  ('C0195', 0.9929070572506444)],
 'C0005': [('C0123', 0.998486142181796),
  ('C0063', 0.9981213043084383),
  ('C0095', 0.9964616563593731)],
 'C0006': [('C0079', 0.9999840493102282),
  ('C0117', 0.9984901469848415),
  ('C0196', 0.9930097360770448)],
 'C0007': [('C0140', 0.9983719163156526),
  ('C0085', 0.996300406654974),
  ('C0080', 0.9926522308907588)],
 'C0008': [('C0090', 0.9880084344632301),
  ('C0179', 0.9868638437014254),
  ('C0084', 0.9744584117911529)],
 'C0009': [('C0192', 0.9964884786651573),
  ('C0083', 0.9924223257792536),
  ('C0043', 0.9921656507553764)],
 'C0010': [('C0029', 

In [9]:
# Convert the lookalike dictionary into a DataFrame for saving
lookalike_list = []
for cust_id, recommendations in lookalike_dict.items():
    for rec in recommendations:
        lookalike_list.append([cust_id, rec[0], rec[1]])

# Create a DataFrame for the Lookalike recommendations
lookalike_df = pd.DataFrame(lookalike_list, columns=['CustomerID', 'RecommendedCustomerID', 'SimilarityScore'])

# Save the recommendations to a CSV file
lookalike_df.to_csv(r'C:\Users\WIN-11\Desktop\Zeotap Internship\data\Lookalike.csv', index=False)


In [10]:
lookalike_df.head(20)

,CustomerID,RecommendedCustomerID,SimilarityScore
0,C0001,C0164,0.971775
1,C0001,C0137,0.954396
2,C0001,C0103,0.946100
3,C0002,C0029,0.999543
4,C0002,C0031,0.997827
5,C0002,C0077,0.995802
6,C0003,C0027,0.838993
7,C0003,C0176,0.827958
8,C0003,C0073,0.765272
9,C0004,C0075,0.997562
